# HARUM NOIR — ZERO COST STUDIO
Pipeline sem assinatura: FFmpeg/CPU + Piper pt-BR + LivePortrait + LTX-Video 2B.


In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg git git-lfs
!pip -q install --upgrade huggingface_hub piper-tts pillow


In [ ]:
from google.colab import files
from pathlib import Path
uploaded=files.upload()
SOURCE=Path('/content')/next(iter(uploaded.keys()))
print(SOURCE)


## NoirMotion — fallback CPU
Funciona mesmo sem GPU.


In [ ]:
import subprocess
from pathlib import Path
DURATION=9; FPS=30
OUT=Path('/content/HARUM_NOIR_NOIRMOTION.mp4')
vf="scale=1200:2134:force_original_aspect_ratio=increase,crop=1200:2134,crop=1080:1920:x='(iw-ow)/2+20*sin(t*0.22)':y='(ih-oh)/2+12*sin(t*0.17)',eq=contrast=1.04:saturation=0.88:brightness=-0.015,vignette=PI/5,noise=alls=2:allf=t+u,fade=t=in:st=0:d=0.35,fade=t=out:st=8.5:d=0.5"
subprocess.run(['ffmpeg','-y','-loop','1','-i',str(SOURCE),'-t',str(DURATION),'-r',str(FPS),'-vf',vf,'-an','-c:v','libx264','-crf','18','-pix_fmt','yuv420p','-movflags','+faststart',str(OUT)],check=True)
print(OUT)


## Piper pt-BR — voz local gratuita


In [ ]:
from huggingface_hub import hf_hub_download
VOICE_DIR=Path('/content/piper_voice'); VOICE_DIR.mkdir(exist_ok=True)
model=hf_hub_download(repo_id='rhasspy/piper-voices',filename='pt/pt_BR/faber/medium/pt_BR-faber-medium.onnx',local_dir=VOICE_DIR)
config=hf_hub_download(repo_id='rhasspy/piper-voices',filename='pt/pt_BR/faber/medium/pt_BR-faber-medium.onnx.json',local_dir=VOICE_DIR)
SCRIPT='Eu quase nunca sei quando um desenho terminou. Às vezes eu só paro antes de estragar.'
VOICE=Path('/content/harum_noir_voice.wav')
subprocess.run(['piper','--model',model,'--config',config,'--output_file',str(VOICE)],input=SCRIPT.encode('utf-8'),check=True)


## LivePortrait — FACE LOCK
Envie um driving video curto e discreto.


In [ ]:
%cd /content
!rm -rf LivePortrait
!git clone -q --depth 1 https://github.com/KlingTeam/LivePortrait.git
%cd /content/LivePortrait
!pip -q install -r requirements.txt
!git lfs install -q
!rm -rf temp_pretrained_weights
!git clone -q https://huggingface.co/KwaiVGI/LivePortrait temp_pretrained_weights
!mkdir -p pretrained_weights
!cp -r temp_pretrained_weights/* pretrained_weights/
!rm -rf temp_pretrained_weights


## LTX-Video 2B — GPU gratuita quando disponível


In [ ]:
%cd /content
!rm -rf LTX-Video
!git clone -q --depth 1 https://github.com/Lightricks/LTX-Video.git
%cd /content/LTX-Video
!pip -q install -e .[inference]


In [ ]:
PROMPT='A quiet nocturnal charcoal atelier. The same woman from the reference remains identical: same adult face, short black chin-length bob, matte black clothing. Restrained documentary realism, graphite dust, ivory paper, no face change.'
subprocess.run(['python','inference.py','--prompt',PROMPT,'--conditioning_media_paths',str(SOURCE),'--conditioning_start_frames','0','--height','768','--width','448','--num_frames','65','--seed','2317','--pipeline_config','configs/ltxv-2b-0.9.8-distilled.yaml','--output_path','/content/ltx_out'],check=True)
